# Instalar dependencias

In [ ]:
!pip install numpy==1.23.5 --force-reinstall

In [ ]:
!pip install ultralytics

# Descargar repositorio Depth Anything V2

In [ ]:
!git clone https://github.com/DepthAnything/Depth-Anything-V2 && \
cd Depth-Anything-V2 && \
pip install -r requirements.txt

# Log in HuggingFace

In [ ]:
from huggingface_hub import login
login()

# Descargar archivo de pesos preentrenados de Deep Learning Depth Anything V2

In [ ]:
!mkdir -p checkpoints
!wget -O checkpoints/depth_anything_v2_vitl.pth https://huggingface.co/depth-anything/Depth-Anything-V2-Large/resolve/main/depth_anything_v2_vitl.pth

# Agregar ruta

In [ ]:
import sys
sys.path.append('/content/Depth-Anything-V2') # Para Collab

# Cargar modelo de profundidad Depth Anything V2

In [ ]:
import cv2
import torch

from depth_anything_v2.dpt import DepthAnythingV2

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

model_configs = {
    'vits': {'encoder': 'vits', 'features': 64, 'out_channels': [48, 96, 192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
    'vitg': {'encoder': 'vitg', 'features': 384, 'out_channels': [1536, 1536, 1536, 1536]}
}

encoder = 'vitl' # or 'vits', 'vitb', 'vitg'

depth_anything = DepthAnythingV2(**model_configs[encoder])
depth_anything.load_state_dict(torch.load(f'checkpoints/depth_anything_v2_{encoder}.pth', map_location=DEVICE))
depth_model = depth_anything.to(DEVICE).eval()

# Obtener el mapeo del video (archivos .npy)

In [ ]:
import cv2
import os
import numpy as np

VIDEO_PATH = "video4.mp4"
OUTPUT_FOLDER = "depth_maps"

# Crear carpeta si no existe
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Cargar video
cap = cv2.VideoCapture(VIDEO_PATH)
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Calcular profundidad
    depth_map = depth_model.infer_image(frame)  # (H, W)

    # Guardar como .npy
    np.save(os.path.join(OUTPUT_FOLDER, f"depth_{frame_count}.npy"), depth_map)

    print(f"Frame {frame_count} procesado y guardado.")
    frame_count += 1

cap.release()
print("✅ Todos los mapas de profundidad guardados.")

# Algoritmo de detección de golpes

In [ ]:
# detect_mano_cabeza_golpe.py
import os
import cv2
import csv
from ultralytics import YOLO

# =================== CONFIG ===================
MODEL_PATH = "best.pt"         # tu modelo entrenado
VIDEO_PATH = "video1.mp4"      # tu video de entrada

OUTPUT_DIR = "salidas_detect"
OUT_VIDEO = os.path.join(OUTPUT_DIR, "video_detectado.mp4")
OUT_CSV   = os.path.join(OUTPUT_DIR, "detecciones.csv")

IMG_SIZE = 1280                # tamaño de inferencia
CONF_THR = 0.25                # umbral de confianza
IOU_THR  = 0.50                # NMS IOU (Ultralytics lo maneja interno)

# Colores por clase (B,G,R)
COLOR_POR_CLASE = {
    "mano":   (0, 255, 255),   # amarillo
    "cabeza": (255, 0, 0),     # azul
    "golpe":  (0, 165, 255),   # naranja
}
TEXTO_COLOR = (255, 255, 255)  # blanco
TEXTO_BG    = (0, 0, 0)        # negro

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =================== CARGA MODELO ===================
model = YOLO(MODEL_PATH)

# =================== VIDEO IN/OUT ===================
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"No se pudo abrir el video: {VIDEO_PATH}")

w  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUT_VIDEO, fourcc, fps, (w, h))

# =================== CSV ===================
csv_file = open(OUT_CSV, "w", newline="", encoding="utf-8")
csv_writer = csv.writer(csv_file)
csv_writer.writerow(["frame", "clase", "confianza", "x1", "y1", "x2", "y2"])

# =================== LOOP ===================
frame_idx = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    frame_idx += 1

    # Inferencia
    # Nota: puedes pasar 'classes=' para filtrar clases si tu modelo tiene más.
    results = model(frame, imgsz=IMG_SIZE, conf=CONF_THR, iou=IOU_THR, verbose=False)[0]

    # Si el modelo tiene nombres: model.names -> {id: "clase"}
    names = model.names if hasattr(model, "names") else None

    if results.boxes is not None:
        boxes_xyxy = results.boxes.xyxy.cpu().numpy()       # (N,4)
        confs      = results.boxes.conf.cpu().numpy()       # (N,)
        clss       = results.boxes.cls.cpu().numpy().astype(int)  # (N,)

        for (x1, y1, x2, y2), conf, cls_id in zip(boxes_xyxy, confs, clss):
            # Nombre de clase
            if names is not None and cls_id in names:
                clase = names[cls_id].strip().lower()
            else:
                # si no hay nombres en el modelo, no filtramos
                clase = str(cls_id)

            # Solo marcamos mano / cabeza / golpe
            if clase not in ("mano", "cabeza", "golpe"):
                continue

            # Dibujo de bbox
            x1i, y1i, x2i, y2i = map(int, [x1, y1, x2, y2])
            color = COLOR_POR_CLASE.get(clase, (0, 255, 0))
            cv2.rectangle(frame, (x1i, y1i), (x2i, y2i), color, 2)

            # Etiqueta
            label = f"{clase} {conf:.2f}"
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(frame, (x1i, max(0, y1i - th - 6)), (x1i + tw + 6, y1i), color, -1)
            cv2.putText(frame, label, (x1i + 3, y1i - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.6, TEXTO_COLOR, 2, cv2.LINE_AA)

            # Log CSV
            csv_writer.writerow([frame_idx, clase, f"{conf:.4f}", x1i, y1i, x2i, y2i])

    # Overlay contador de detecciones en el frame
    total_det = 0 if results.boxes is None else len(results.boxes)
    cv2.putText(frame, f"Detecciones: {total_det}", (16, 32), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (36, 255, 102), 2)

    writer.write(frame)

# =================== CIERRE ===================
cap.release()
writer.release()
csv_file.close()

print(f"✅ Video anotado: {OUT_VIDEO}")
print(f"✅ CSV detecciones: {OUT_CSV}")


# Algortimo de calculo de velocidad + detección

In [ ]:
from ultralytics import YOLO
import cv2, os, math, glob
import numpy as np
from collections import deque

# ===================== CONFIG =====================
MODEL_PATH = "best.pt"
VIDEO_PATH = "video1.mp4"
DEPTH_MAPS_FOLDER = "depth_maps"

OUTPUT_VIDEO_PATH = "video_salida/video_salida.mp4"
OUTPUT_FRAMES_DIR = "frames_golpe"
LOG_FILE = "log_golpes.txt"
DEBUG_CSV_PATH = "debug_deltas.csv"
RESULTADOS_PATH = "resultados.txt"  

# Nombres de clases
CLASE_GOLPE  = "golpe"
CLASE_CABEZA = "cabeza"
CLASE_MANO   = "mano"

# Umbrales / YOLO
CONFIDENCE_THRESHOLD = 0.25
IOU_NMS_THRESHOLD    = 0.50
IMG_SIZE             = 1280

# Tracking y conteo
DIST_THRESH_PX       = 40
MIN_STREAK           = 1
MAX_AGE              = 6
COOLDOWN_PER_TRACK   = 8
SAVE_CROPS           = True

# --------- Velocidad / Profundidad ---------
AUTO_MAX_REAL_DISTANCE = 2.0     # ↓ (ayuda a bajar v)
MAX_VALID_VELOCITY     = 10.0    # si se pasa, se clampa a este valor
VENTANA_FRAMES         = 3
MIN_DETECCIONES_EN_VENTANA = 2

# Asociación mano↔golpe
MANO_MAX_FRAMES  = 55
MANO_MAX_DISTPX  = 100

# Profundidad
DEPTH_TOLERANCE_FRAMES = 2
WINDOW_MEDIAN           = 3  # 7x7 para más robustez

# Física
MAX_PENETRATION_M = 0.35   # 35 cm; no debería superarse en un golpe

# ===================== UTILS =====================
def iou(a,b):
    xA=max(a[0],b[0]); yA=max(a[1],b[1]); xB=min(a[2],b[2]); yB=min(a[3],b[3])
    inter=max(0,xB-xA+1)*max(0,yB-yA+1)
    areaA=(a[2]-a[0]+1)*(a[3]-a[1]+1); areaB=(b[2]-b[0]+1)*(b[3]-b[1]+1)
    union=areaA+areaB-inter
    return inter/union if union>0 else 0.0

def nms(boxes, scores, thr):
    idxs=sorted(range(len(boxes)), key=lambda i: scores[i], reverse=True)
    keep=[]
    while idxs:
        i=idxs.pop(0); keep.append(i)
        idxs=[j for j in idxs if iou(boxes[i], boxes[j])<thr]
    return keep

def center(box):
    x1,y1,x2,y2=box; return ((x1+x2)//2, (y1+y2)//2)

def d2(a,b): return math.hypot(a[0]-b[0], a[1]-b[1])

def suavizar_profundidad(mapa, cx, cy, r=WINDOW_MEDIAN):
    h, w = mapa.shape
    x1, x2 = max(0, cx - r), min(w, cx + r + 1)
    y1, y2 = max(0, cy - r), min(h, cy + r + 1)
    return float(np.nanmedian(mapa[y1:y2, x1:x2]))

# ===================== DEPTH HELPERS (escala global) =====================
GLOBAL_SCENE_MIN, GLOBAL_SCENE_MAX = None, None

def init_global_depth_stats(folder, sample_limit=200):
    files = sorted(glob.glob(os.path.join(folder, "depth_*.npy")))[:sample_limit]
    if not files:
        return None, None
    vals = []
    for fp in files:
        dm = np.load(fp)
        sub = dm[::8, ::8].ravel()  # muestreo
        vals.append(sub)
    vals = np.concatenate(vals)
    gmin = np.nanpercentile(vals, 2)
    gmax = np.nanpercentile(vals, 98)
    return float(gmin), float(gmax)

def depth_path_for_frame(idx: int):
    candidates = [
        os.path.join(DEPTH_MAPS_FOLDER, f"depth_{idx}.npy"),
        os.path.join(DEPTH_MAPS_FOLDER, f"depth_{idx:04d}.npy"),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    for d in range(1, DEPTH_TOLERANCE_FRAMES+1):
        for cand_idx in (idx-d, idx+d):
            for p in (os.path.join(DEPTH_MAPS_FOLDER, f"depth_{cand_idx}.npy"),
                      os.path.join(DEPTH_MAPS_FOLDER, f"depth_{cand_idx:04d}.npy")):
                if os.path.exists(p):
                    return p
    return None

def load_depth_map(idx:int):
    dp = depth_path_for_frame(idx)
    if not dp:
        return None
    return np.load(dp)

def ensure_global_scale(dm):
    global GLOBAL_SCENE_MIN, GLOBAL_SCENE_MAX
    if GLOBAL_SCENE_MIN is None or GLOBAL_SCENE_MAX is None:
        gmin, gmax = init_global_depth_stats(DEPTH_MAPS_FOLDER, sample_limit=200)
        if gmin is not None and gmax is not None and gmax > gmin:
            GLOBAL_SCENE_MIN, GLOBAL_SCENE_MAX = gmin, gmax
        else:
            GLOBAL_SCENE_MIN = float(np.nanpercentile(dm, 2))
            GLOBAL_SCENE_MAX = float(np.nanpercentile(dm, 98))

def safe_depth_to_meters(raw_depth, scene_min, scene_max):
    # 1) clamp raw al rango de escena
    raw_clamped = np.clip(raw_depth, scene_min, scene_max)
    # 2) normalizar y clipear a [0,1] para evitar negativos / >1
    norm = (scene_max - raw_clamped) / max(scene_max - scene_min, 1e-6)
    norm = float(np.clip(norm, 0.0, 1.0))
    # 3) escalar a metros
    return norm * AUTO_MAX_REAL_DISTANCE

def temporal_depth_median(cx, cy, frame_idx, radius=1):
    """Mediana temporal en t-1..t..t+1 (si existen maps)."""
    vals = []
    for dt in range(-radius, radius+1):
        dm = load_depth_map(frame_idx + dt)
        if dm is None: 
            continue
        if 0 <= cy < dm.shape[0] and 0 <= cx < dm.shape[1]:
            # aseguramos escala global
            ensure_global_scale(dm)
            d = suavizar_profundidad(dm, cx, cy, r=WINDOW_MEDIAN)
            vals.append(safe_depth_to_meters(d, GLOBAL_SCENE_MIN, GLOBAL_SCENE_MAX))
    if not vals:
        return None
    return float(np.nanmedian(vals))

# ===================== TRACKER =====================
class Track:
    _next_id=1
    def __init__(self, cx, cy, box, frame_idx):
        self.id=Track._next_id; Track._next_id+=1
        self.cx, self.cy = cx, cy
        self.box = box
        self.last_seen = frame_idx
        self.streak = 1
        self.confirmed = False
        self.last_count_frame = -10**9
        self.history = deque(maxlen=30)
        self.history.append((cx,cy,frame_idx))
    def update(self, cx, cy, box, frame_idx, consecutive=True):
        self.cx, self.cy = cx, cy
        self.box = box
        self.history.append((cx,cy,frame_idx))
        self.streak = self.streak+1 if consecutive else 1
        self.last_seen = frame_idx

# ===================== PREP =====================
os.makedirs(os.path.dirname(OUTPUT_VIDEO_PATH), exist_ok=True)
os.makedirs(OUTPUT_FRAMES_DIR, exist_ok=True)

model = YOLO(MODEL_PATH)
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"No se pudo abrir {VIDEO_PATH}")

w=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); h=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps=cap.get(cv2.CAP_PROP_FPS) or 25.0
frame_time = 1.0/float(fps)
MIN_DELTA_T = max(0.9*frame_time, 0.03)

fourcc=cv2.VideoWriter_fourcc(*"mp4v")
writer=cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (w,h))

# logs
log=open(LOG_FILE,"w",encoding="utf-8")
log.write("id_evento,frame,id_track,velocidad_m_s,intensidad,razon,x1,y1,x2,y2\n")
dbg=open(DEBUG_CSV_PATH,"w",encoding="utf-8")
dbg.write("frame_evento,mano_frame,cabeza_frame,depth_mano,depth_cabeza,delta_d,delta_t,velocidad,intensidad,razon\n")
res = open(RESULTADOS_PATH, "w", encoding="utf-8", buffering=1)
res.write("id_evento,velocidad_m_s,intensidad,frame\n")

# estado
tracks=[]; frame_idx=0; total=0
ventana_golpes = deque([0]*VENTANA_FRAMES, maxlen=VENTANA_FRAMES)
evento_en_ventana = False

# historial de manos
historial_manos = {}

# ===================== LOOP =====================
while True:
    ok, frame = cap.read()
    if not ok: break
    frame_idx += 1

    res_yolo = model(frame, imgsz=IMG_SIZE, conf=CONFIDENCE_THRESHOLD, verbose=False)[0]

    golpes_raw=[]; scores_g=[]
    manos_pts=[]; cabezas_pts=[]
    for b in res_yolo.boxes:
        cls_id=int(b.cls[0]); label=model.names[cls_id].strip().lower()
        conf=float(b.conf[0])
        x1,y1,x2,y2=map(int, b.xyxy[0])
        x1,y1=max(0,x1),max(0,y1); x2,y2=min(w-1,x2),min(h-1,y2)
        if x2<=x1 or y2<=y1 or conf<CONFIDENCE_THRESHOLD: 
            continue
        if label==CLASE_GOLPE:
            golpes_raw.append((x1,y1,x2,y2)); scores_g.append(conf)
        elif label==CLASE_MANO:
            manos_pts.append(((x1+x2)//2, (y1+y2)//2))
        elif label==CLASE_CABEZA:
            cabezas_pts.append(((x1+x2)//2, (y1+y2)//2))

    keep = nms(golpes_raw, scores_g, IOU_NMS_THRESHOLD) if golpes_raw else []
    boxes = [golpes_raw[i] for i in keep]
    centers = [center(b) for b in boxes]

    # ---- actualizar historial manos con profundidad (temporal mediana) ----
    for (mx,my) in manos_pts:
        d_mano = temporal_depth_median(mx, my, frame_idx, radius=1)
        historial_manos[f"{mx}_{my}"] = {"frame": frame_idx, "depth": d_mano, "cx": mx, "cy": my}

    # ------ asociar a pistas ------
    unmatched=set(range(len(boxes)))
    for trk in list(tracks):
        if frame_idx - trk.last_seen > MAX_AGE:
            tracks.remove(trk); continue
        best=None; bestd=1e9
        for j in list(unmatched):
            d = d2((trk.cx,trk.cy), centers[j])
            if d < bestd and d <= DIST_THRESH_PX:
                bestd=d; best=j
        if best is not None:
            trk.update(centers[best][0], centers[best][1], boxes[best],
                       frame_idx, consecutive=(frame_idx==trk.last_seen+1))
            unmatched.remove(best)
    for j in unmatched:
        cx,cy = centers[j]
        tracks.append(Track(cx,cy,boxes[j],frame_idx))

    golpe_detectado_en_frame = 1 if len(boxes)>0 else 0
    ventana_golpes.append(golpe_detectado_en_frame)
    evento_en_ventana = (sum(ventana_golpes) >= MIN_DETECCIONES_EN_VENTANA)

    for trk in list(tracks):
        x1,y1,x2,y2 = trk.box
        cv2.rectangle(frame,(x1,y1),(x2,y2),(0,220,255),2)
        cv2.putText(frame,f"T{trk.id} s{trk.streak}",(x1,max(20,y1-6)),cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,220,255),2)

        if (not trk.confirmed) and trk.streak >= MIN_STREAK and (frame_idx - trk.last_count_frame) >= COOLDOWN_PER_TRACK:
            if not evento_en_ventana:
                continue

            gx, gy = trk.cx, trk.cy

            # cabeza más cercana (mediana temporal)
            cabeza = None
            if cabezas_pts:
                cabeza = min(cabezas_pts, key=lambda c: math.hypot(c[0]-gx, c[1]-gy))
                cxh, cyh = cabeza
                d_cabeza = temporal_depth_median(cxh, cyh, frame_idx, radius=1)
            else:
                cxh, cyh = gx, gy
                d_cabeza = temporal_depth_median(cxh, cyh, frame_idx, radius=1)

            # mano previa (max dt dentro del radio)
            mejor_mano = None
            mejor_dt = 0.0
            for mid, data in historial_manos.items():
                mx, my = data["cx"], data["cy"]
                dist = math.hypot(mx-gx, my-gy)
                df = frame_idx - data["frame"]
                if 0 < df <= MANO_MAX_FRAMES and dist < MANO_MAX_DISTPX:
                    dt = df / fps
                    if dt > mejor_dt:
                        mejor_dt = dt
                        mejor_mano = data

            intensidad = "desconocida"
            razon = ""
            velocidad = 0.0
            delta_d = None
            delta_t = mejor_dt

            if mejor_mano is None:
                razon = "sin_mano"
            elif d_cabeza is None:
                razon = "sin_depth_cabeza"
            else:
                d_mano = mejor_mano["depth"]
                if d_mano is None:
                    # temporal mediana en el frame de la mano
                    d_mano = temporal_depth_median(mejor_mano["cx"], mejor_mano["cy"], mejor_mano["frame"], radius=1)

                if d_mano is None:
                    razon = "sin_depth_mano"
                elif delta_t <= 0:
                    razon = "delta_t_cero"
                else:
                    delta_d = d_mano - d_cabeza

                    # Gating lógico: si mano está más lejos que cabeza en el frame del evento (no hay acercamiento real)
                    if d_mano < d_cabeza - 0.03:  # 3cm tolerancia
                        razon_local = "mano_no_mas_cercana"
                    else:
                        razon_local = "ok"

                    # Clamp físico de penetración
                    if abs(delta_d) > MAX_PENETRATION_M:
                        delta_d = math.copysign(MAX_PENETRATION_M, delta_d)
                        razon = "clamped_delta_d"
                    else:
                        if razon == "": razon = razon_local

                    v = abs(delta_d) / delta_t

                    if delta_t < MIN_DELTA_T:
                        # mantener v, marcar como tiempo_bajo
                        velocidad = min(v, MAX_VALID_VELOCITY)
                        intensidad = ("leve" if velocidad < 0.5 else "media" if velocidad < 1.5 else "fuerte")
                        if razon == "": razon = "tiempo_bajo"
                    elif v > MAX_VALID_VELOCITY:
                        velocidad = MAX_VALID_VELOCITY
                        intensidad = "fuerte"
                        if razon == "": razon = "clamped_velocidad_alta"
                    else:
                        velocidad = v
                        if razon == "": razon = "ok"
                        if v < 0.5:   intensidad = "leve"
                        elif v < 1.5: intensidad = "media"
                        else:         intensidad = "fuerte"

            # debug
            dbg.write(
                f"{frame_idx},{mejor_mano['frame'] if mejor_mano else -1},{frame_idx if cabeza else -1},"
                f"{(mejor_mano['depth'] if (mejor_mano and mejor_mano['depth'] is not None) else float('nan')):.4f},"
                f"{(d_cabeza if d_cabeza is not None else float('nan')):.4f},"
                f"{(delta_d if delta_d is not None else float('nan')):.4f},"
                f"{(delta_t if delta_t is not None else float('nan')):.4f},"
                f"{velocidad:.4f},{intensidad},{razon}\n"
            )

            # confirmar evento
            total += 1
            trk.confirmed = True
            trk.last_count_frame = frame_idx

            if SAVE_CROPS:
                crop = frame[max(0,y1):min(h,y2), max(0,x1):min(w,x2)]
                cv2.imwrite(os.path.join(OUTPUT_FRAMES_DIR, f"golpe_{total:04d}_f{frame_idx}_T{trk.id}.jpg"), crop)

            log.write(f"{total},{frame_idx},{trk.id},{velocidad:.3f},{intensidad},{razon},{x1},{y1},{x2},{y2}\n")

            res.write(f"{total},{velocidad:.3f},{intensidad},{frame_idx}\n")

        # rastro
        for k in range(1, len(trk.history)):
            x0,y0,_ = trk.history[k-1]
            x1h,y1h,_ = trk.history[k]
            cv2.line(frame, (x0,y0), (x1h,y1h), (0,220,255), 2)

    cv2.putText(frame, f"Golpes: {total}", (16,32), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (36,255,102), 2)
    writer.write(frame)

    historial_manos = {k:v for k,v in historial_manos.items() if frame_idx - v["frame"] <= MANO_MAX_FRAMES}

# ===================== CIERRE =====================
cap.release(); writer.release(); log.close(); dbg.close(); res.close(); cv2.destroyAllWindows()
print("FPS:", fps, "| MIN_DELTA_T:", round(MIN_DELTA_T,4))
print("Global depth min/max:", GLOBAL_SCENE_MIN, GLOBAL_SCENE_MAX)
print("Total golpes:", total)
print(f"Video anotado: {OUTPUT_VIDEO_PATH}")
print(f"Log eventos:   {LOG_FILE}")
print(f"Debug CSV:     {DEBUG_CSV_PATH}")
print(f"Resultados:    {RESULTADOS_PATH}")
print(f"Crops:         {OUTPUT_FRAMES_DIR}")